# 00 — CSV → Parquet

**Цель:** посмотреть исходный файл Lending Club и сохранить его в Parquet — рабочую копию, с которой дальше работаем.

- **Вход:** `data/raw/accepted_2007_to_2018Q4.csv.gz` (сжатый CSV, 374 МБ; распакованный — 1,68 ГБ)
- **Выход:** `data/accepted_2007_to_2018Q4.parquet` (258 МБ)
- **Инструмент:** DuckDB — база данных внутри Python, выполняет SQL прямо по файлам, без сервера и без загрузки всего файла в память.

**Принцип:** Parquet — *сырая* копия CSV. Типы не исправляем, строки не удаляем — очистка будет в отдельном коде (`src/`), чтобы каждое решение было видно и проверяемо.

**Итог:** 2 260 701 строка, 151 колонка, строки в CSV и Parquet совпадают.

## 1. Настройка

`duckdb.connect()` создаёт базу в памяти — на диск ничего не пишется. `display.max_rows` разрешает pandas показывать до 200 строк, иначе длинные таблицы обрезаются.

DuckDB читает `.csv.gz` напрямую, распаковывать не нужно. Путь начинается с `../`, потому что ноутбук запускается из папки `notebooks/`.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)

In [ ]:
# In-memory database; DuckDB reads .csv.gz directly
con = duckdb.connect()
raw = "../data/raw/accepted_2007_to_2018Q4.csv.gz"


## 2. Какие колонки и типы угадал DuckDB

`DESCRIBE` показывает название и тип каждой колонки. Типы DuckDB **угадывает** — по умолчанию только по первым ~20 тыс. строк, а не по всему файлу (это важно, см. раздел 4).

In [ ]:
# Column names and types inferred by DuckDB (from a sample of rows)
cols_described = con.sql(f"DESCRIBE SELECT * FROM '{raw}'").df()

In [ ]:
cols_described

### Колонки, в типах которых были сомнения

`.isin()` проверяет каждую строку и возвращает колонку `True`/`False` (булева маска); по ней pandas оставляет нужные строки. Обычный `in` здесь не работает — он ждёт один ответ для всей колонки.

| Колонка | Тип | Вывод |
|---|---|---|
| `int_rate`, `revol_util` | DOUBLE | уже числа, всё в порядке |
| `term` | VARCHAR | срок текстом (`36 months`) → нужно число месяцев |
| `emp_length` | VARCHAR | стаж текстом (`10+ years`, `< 1 year`) → число лет |
| `issue_d` | VARCHAR | месяц выдачи текстом (`Mar-2017`) → дата |

In [ ]:
some_columns = ('int_rate', 'issue_d', 'emp_length', 'term','revol_util')
cols_described[cols_described['column_name'].isin(some_columns)]
#cols_described['column']

## 3. Значения в текстовых колонках

`GROUP BY` собирает строки с одинаковым значением в группу, `count(*)` считает их, `ORDER BY n DESC` сортирует по убыванию.

**`term`:** 36 месяцев — 1 609 754 (71%), 60 месяцев — 650 914, **пусто — 33** (разбираемся в разделе 4). Возможно, в начале значения есть пробел (`" 36 months"`) — pandas его не показывает; проверить через `length(term)`.

In [ ]:
con.sql(f"""
    SELECT term, count(*) AS n
    FROM '{raw}'
    GROUP BY term
    ORDER BY n DESC
""").df()

**`emp_length`:**
- 146 940 пропусков (6,5%) — это информация, а не мусор: «не работает» или «не указал». Нельзя просто заменить на 0 — смешается с `< 1 year` (190 тыс.). Гипотеза для `hypotheses.md`: отличается ли доля дефолтов у заёмщиков с пропуском.
- `10+ years` — треть заёмщиков: Lending Club обрезал стаж сверху, 10 и 30 лет неразличимы.

In [ ]:
con.sql(f"""
    SELECT emp_length, count(*) AS n
    FROM '{raw}'
    GROUP BY emp_length
    ORDER BY n DESC
""").df()

**`issue_d`:** формат везде `Mon-YYYY` (например, `Mar-2015`), годы 2007–2018. В дату переводится через `strptime(issue_d, '%b-%Y')`.

In [ ]:
con.sql(f"""
    SELECT DISTINCT issue_d
    FROM '{raw}'
    LIMIT 20
""").df()

## 4. 33 служебные строки

Первая попытка этого запроса упала с ошибкой: DuckDB по первым 20 480 строкам решил, что `id` — целое число (BIGINT), а в строке 421 097 встретил текст `Total amount funded in policy code 1: ...`.

**Почему раньше не падало:** DuckDB разбирает только колонки, нужные запросу (*projection pushdown*). Прошлые запросы не трогали `id`.

**Решение:** `read_csv(..., types={'id': 'VARCHAR'})` — задать тип `id` вручную как текст. Двойные `{{ }}` — из-за f-строки: так Python оставляет обычные фигурные скобки.

**Что нашли:** это не займы, а остатки склейки 16 квартальных файлов Lending Club — 16 пар итогов (`policy code 1` / `policy code 2`) и заголовок раздела `Loans that do not meet the credit policy`.

**Решение (записано в `decisions.md`, 2026-10-12):** в сыром Parquet строки оставляем, удаляем на шаге очистки правилом `TRY_CAST(id AS BIGINT) IS NULL`.

In [ ]:
# id is read as text: the file contains non-loan service rows with text in id
con.sql(f"SELECT id, loan_amnt, issue_d, loan_status FROM read_csv('{raw}', types={{'id': 'VARCHAR'}}) WHERE term IS NULL").df()

## 5. Конвертация в Parquet

- `memory_limit = '3GB'` — DuckDB не возьмёт больше 3 ГБ памяти (на ноутбуке 8 ГБ); если не хватит — временно сбросит данные на диск, а не упадёт.
- `COPY (запрос) TO 'файл'` — пишет результат запроса в файл. CSV читается потоком, по кускам, а не целиком в память.
- `types = {'id': 'VARCHAR'}` — из-за служебных строк (раздел 4).
- `sample_size = -1` — угадывать типы по **всему** файлу, а не по первым 20 тыс. строк. Медленнее (файл читается дважды), но такая же ошибка, как с `id`, не случится в другой колонке.
- `COMPRESSION zstd` — сжатие: лучше стандартного snappy при почти той же скорости чтения.

In [ ]:
parquet_path = "../data/accepted_2007_to_2018Q4.parquet"

# Keep DuckDB within a safe share of 8 GB RAM; spill to disk if needed
con.sql("SET memory_limit = '3GB'")

In [ ]:
con.sql(f"""
    COPY (
        SELECT *
        FROM read_csv(
            '{raw}',
            types = {{'id': 'VARCHAR'}},  -- service rows contain text in id
            sample_size = -1              -- scan the whole file to infer types
        )
    )
    TO '{parquet_path}' (FORMAT parquet, COMPRESSION zstd)
""")

## 6. Проверка

Сравниваем число строк в исходнике и в Parquet. CSV считаем с `all_varchar = true` (всё как текст), чтобы подсчёт не зависел от угаданных типов. `.fetchone()[0]` достаёт из результата одно число.

**Результат:** 2 260 701 = 2 260 701, 151 колонка. Размер: 258 МБ против 1,68 ГБ у CSV — в 6,5 раз меньше. Parquet хранит каждую колонку отдельно, поэтому запросы читают только нужные колонки и работают быстрее.

In [ ]:
# Row count check: raw file vs Parquet
rows_csv = con.sql(f"SELECT count(*) FROM read_csv('{raw}', all_varchar = true)").fetchone()[0]
rows_pq = con.sql(f"SELECT count(*) FROM '{parquet_path}'").fetchone()[0]
n_cols = len(con.sql(f"DESCRIBE SELECT * FROM '{parquet_path}'").df())

print(f"rows: csv={rows_csv:,}  parquet={rows_pq:,}  match={rows_csv == rows_pq}")
print(f"columns: {n_cols}")

In [ ]:
# File size on disk
!ls -lh ../data/*.parquet

## 7. Типы в Parquet

- **Даты остались текстом** (`issue_d`, `earliest_cr_line`, `last_pymnt_d`, ...) — как и решили, разбор будет при очистке.
- **Счётчики — DOUBLE** (`delinq_2yrs`, `open_acc`, `policy_code`, ...), хотя по смыслу целые: в CSV числа записаны как `1.0`. Для анализа не мешает.
- **`member_id` — VARCHAR**: подозрение, что колонка пустая.

In [ ]:
# Types stored in Parquet
con.sql(f"DESCRIBE SELECT * FROM '{parquet_path}'").df()

`count(колонка)` считает только непустые значения. Результат **0** — `member_id` полностью пустая (Lending Club убрал идентификатор заёмщика из выгрузки). Кандидат на исключение при разметке колонок.

In [ ]:
# count(col) counts non-null values only
con.sql(f"SELECT count(member_id) FROM '{parquet_path}'").df()

## Итоги и следующие шаги

**Сделано:** рабочая копия `data/accepted_2007_to_2018Q4.parquet`, проверена на совпадение строк. Распакованный CSV удалён, резерв — `data/raw/*.csv.gz`.

**Найдено:**
- 33 служебные строки → удаляем при очистке (`decisions.md`)
- `term`, `emp_length`, даты — текст, нужен разбор
- `emp_length`: 6,5% пропусков — проверить связь с дефолтом (на 2007–2013)
- `member_id` пустая

**Дальше (неделя 1, `01_overview.ipynb`):** объёмы по годам выдачи, доли 36/60, пропуски по колонкам, `policy_code`, разметка колонок по словарю (`docs/LCDataDictionary.xlsx`): «известна при подаче» / «появляется после выдачи».